# Multi-dataset IoT/IIoT IDS experiment

This Kaggle notebook implements the planned experiment for the paper:

> Explanation faithfulness and stability under adversarial evasion in an IoT/IIoT intrusion detection system.

It uses two datasets independently, five model-training seeds per dataset and training condition, a reproduced MLP baseline, constrained PGD/FGSM evasion, SHAP/LIME explanations, seed-level statistical testing, and Springer-compatible PDF figures.

## Dataset links

### ToN-IoT
- Official UNSW dataset page: <https://research.unsw.edu.au/projects/toniot-datasets>
- Kaggle network subset: <https://www.kaggle.com/datasets/arnobbhowmik/ton-iot-network-dataset>
- Alternative Kaggle copy: <https://www.kaggle.com/datasets/alaaelmor/ton-iot-train-test-network>

### Edge-IIoTset
- IEEE DataPort page: <https://ieee-dataport.org/documents/edge-iiotset-new-comprehensive-realistic-cyber-security-dataset-iot-and-iiot-applications>
- Kaggle dataset: <https://www.kaggle.com/datasets/mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot>
- Published dataset paper DOI: <https://doi.org/10.1109/ACCESS.2022.3165809>

For the free-cloud experiment, use `Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv` rather than the much larger DNN table. The discovery cell prefers this smaller merged ML table automatically.

The notebook is configured for the two paths currently visible in your Kaggle session:

```text
/kaggle/input/datasets/smnihalahmed/dataset-1/Train_Test_datasets/Train_Test_Network_dataset/train_test_network.csv
/kaggle/input/datasets/mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot/Edge-IIoTset dataset/Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv
```

Add both datasets through **Add Input**. The exact paths are set in `MANUAL_FILES`. If Kaggle assigns different input paths later, set those two values to the new paths or change them back to `None` to enable recursive discovery.

## Stage outputs

Every stage writes its outputs under `/kaggle/working/` and immediately creates a downloadable ZIP link using `IPython.display.FileLink`. The final stage creates `final_experiment_bundle.zip` containing the stage ZIPs, CSV tables, JSON metadata, model checkpoints, and PDF figures.

The default configuration is deliberately bounded for a free Kaggle T4/P100. Increase the row and explanation caps only after a pilot run.

In [ ]:
# Stage 0 — install/import configuration
# Kaggle normally already includes torch, sklearn, scipy, matplotlib, pandas, and shap.
# LIME is installed only when absent.
import importlib.util, subprocess, sys
if importlib.util.find_spec("lime") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lime"])


In [ ]:
from pathlib import Path
import json, os, re, sys, time, random, shutil, zipfile, hashlib, warnings, platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display, FileLink, Markdown

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, average_precision_score, confusion_matrix, roc_auc_score
)
from scipy.stats import wilcoxon

import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")

# -------------------- user-adjustable configuration --------------------
ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
SEEDS = [17, 29, 41, 53, 67]       # exactly five model-training seeds
SPLIT_SEED = 2026
MAX_ROWS_PER_DATASET = 500_000      # deterministic cap; set None to use all rows
EXPLAIN_BENIGN_ROWS = 300
EXPLAIN_MALICIOUS_ROWS = 300
ATTACK_ROWS = 300                   # selected from the explanation malicious subset
MAX_EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 2048
LEARNING_RATE = 1e-3
HIDDEN_DIMS = [128, 64, 32]
DROPOUT = 0.20
PGD_STEPS = 10
PGD_RANDOM_STARTS = 1
EPSILONS = [0.01, 0.03, 0.05]
LIME_NUM_SAMPLES = 300
SHAP_BACKGROUND_ROWS = 100
RUN_ADVERSARIAL_TRAINING = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set an exact file path here if automatic discovery selects the wrong CSV.
MANUAL_FILES = {
    "ton": (
        "/kaggle/input/datasets/smnihalahmed/dataset-1/"
        "Train_Test_datasets/Train_Test_Network_dataset/"
        "train_test_network.csv"
    ),
    "edge": (
        "/kaggle/input/datasets/mohamedamineferrag/"
        "edgeiiotset-cyber-security-dataset-of-iot-iiot/"
        "Edge-IIoTset dataset/Selected dataset for ML and DL/"
        "ML-EdgeIIoT-dataset.csv"
    ),
}

DIRS = {
    "stage00": ROOT / "stage_00_environment",
    "processed": ROOT / "processed_datasets",
    "models": ROOT / "models",
    "attacks": ROOT / "attacks",
    "explanations": ROOT / "explanations",
    "figures": ROOT / "figures_pdf",
    "tables": ROOT / "tables",
    "stats": ROOT / "statistics",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "seeds": SEEDS,
    "split_seed": SPLIT_SEED,
    "max_rows_per_dataset": MAX_ROWS_PER_DATASET,
    "explain_benign_rows": EXPLAIN_BENIGN_ROWS,
    "explain_malicious_rows": EXPLAIN_MALICIOUS_ROWS,
    "attack_rows": ATTACK_ROWS,
    "hidden_dims": HIDDEN_DIMS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "pgd_steps": PGD_STEPS,
    "epsilons": EPSILONS,
    "lime_num_samples": LIME_NUM_SAMPLES,
    "device": str(DEVICE),
}
(DIRS["stage00"] / "config.json").write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))


In [ ]:
# Shared utilities used by every stage.
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


RESOURCE_ROWS = []


def _peak_rss_mb():
    """Return process peak resident memory in MiB using the platform resource API."""
    try:
        import resource as _resource
        raw = float(_resource.getrusage(_resource.RUSAGE_SELF).ru_maxrss)
        # Linux reports KiB; macOS reports bytes.
        return raw / (1024.0 if platform.system() != "Darwin" else 1024.0 ** 2)
    except Exception:
        return np.nan


def record_resource(stage: str, output_dir: Path | None = None):
    """Persist explicit process/GPU peak-resource metrics for the current stage."""
    row = {
        "stage": stage,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "device": str(DEVICE),
        "process_peak_rss_mb": _peak_rss_mb(),
        "gpu_peak_allocated_mb": (float(torch.cuda.max_memory_allocated() / 1024**2)
                                   if torch.cuda.is_available() else np.nan),
        "gpu_peak_reserved_mb": (float(torch.cuda.max_memory_reserved() / 1024**2)
                                  if torch.cuda.is_available() else np.nan),
        "cpu_count": int(os.cpu_count() or 1),
    }
    RESOURCE_ROWS.append(row)
    frame = pd.DataFrame(RESOURCE_ROWS)
    frame.to_csv(ROOT / "resource_metrics.csv", index=False)
    if output_dir is not None:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        frame.to_csv(Path(output_dir) / "resource_metrics.csv", index=False)
    return row


def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def zip_stage(stage_name: str, paths):
    """Zip stage outputs and display a clickable Kaggle download link."""
    paths = [Path(p) for p in paths if Path(p).exists()]
    zip_path = ROOT / f"{stage_name}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in paths:
            if p.is_dir():
                for child in p.rglob("*"):
                    if child.is_file():
                        zf.write(child, arcname=str(child.relative_to(ROOT)))
            else:
                zf.write(p, arcname=str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else p.name)
    display(FileLink(str(zip_path), result_html_prefix=f"<b>Download {zip_path.name}:</b> "))
    return zip_path


def write_manifest(directory: Path, name: str, extra=None):
    files = []
    for p in directory.rglob("*"):
        if p.is_file():
            files.append({"path": str(p.relative_to(directory)), "bytes": p.stat().st_size, "sha256": sha256_file(p)})
    manifest = {"stage": name, "created_utc": datetime.now(timezone.utc).isoformat(), "files": files}
    if extra:
        manifest.update(extra)
    path = directory / "manifest.json"
    path.write_text(json.dumps(manifest, indent=2))
    return path


def choose_label_column(columns):
    low = {c.lower(): c for c in columns}
    preferred = ["label", "attack_label", "class", "target", "is_attack", "attack"]
    for p in preferred:
        if p in low:
            return low[p]
    for c in columns:
        if any(k in c.lower() for k in ["label", "class", "target"]):
            return c
    return columns[-1]


def choose_family_column(columns, label_col):
    for c in columns:
        if c == label_col:
            continue
        if c.lower() in {"type", "attack_type", "attack", "category", "attack_category"}:
            return c
    return label_col


def label_to_binary(series):
    if pd.api.types.is_numeric_dtype(series):
        values = pd.to_numeric(series, errors="coerce").fillna(0)
        unique = set(values.unique().tolist())
        if unique.issubset({0, 1}):
            return values.astype(int).to_numpy()
    s = series.astype(str).str.strip().str.lower()
    benign = {"normal", "benign", "0", "false", "non-attack", "nonattack"}
    return (~s.isin(benign)).astype(int).to_numpy()


def find_csvs():
    return sorted(INPUT_ROOT.rglob("*.csv"))


def choose_input(kind):
    if MANUAL_FILES.get(kind):
        p = Path(MANUAL_FILES[kind])
        if not p.exists():
            raise FileNotFoundError(f"Manual path does not exist: {p}")
        return p
    candidates = find_csvs()
    if kind == "ton":
        preferred = [p for p in candidates if "train_test_network" in p.name.lower() or "ton_iot" in p.name.lower() or "ton-iot" in p.name.lower()]
    else:
        # Use the smaller selected ML table by default. It contains the merged
        # Edge-IIoTset records needed here and is much more practical on a free
        # Kaggle runtime than the 1+ GB DNN table.
        ml_preferred = [p for p in candidates if p.name.lower() == "ml-edgeiiot-dataset.csv"]
        if ml_preferred:
            return max(ml_preferred, key=lambda p: p.stat().st_size)
        preferred = [p for p in candidates if "dnn-edgeiiot" in p.name.lower() or "edgeiiot" in p.name.lower() or "edge-iiot" in p.name.lower()]
    if not preferred:
        raise FileNotFoundError(f"No candidate CSV found for {kind}. CSVs seen: {candidates[:20]}")
    # Fall back to the largest matching merged table only if the ML table is absent.
    return max(preferred, key=lambda p: p.stat().st_size)


def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))

print("Device:", DEVICE)
print("CSV candidates under /kaggle/input:")
for p in find_csvs()[:30]:
    print(" ", p)


In [ ]:
# Stage 0 — zip configuration and environment metadata
(DIRS["stage00"] / "environment.json").write_text(json.dumps({
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "shap": shap.__version__,
}, indent=2))
record_resource("stage_00_environment", DIRS["stage00"])
write_manifest(DIRS["stage00"], "stage_00_environment")
zip_stage("stage_00_environment", [DIRS["stage00"]])


## Stage 1 — discover, audit, preprocess, and freeze datasets

This stage does not pool the datasets. It selects the ToN-IoT network CSV and the Edge-IIoTset selected network CSV, removes labels and provenance/identifier fields, keeps numerical traffic features, creates one fixed stratified partition per dataset, and saves the exact evaluation record IDs.

In [ ]:
# Stage 1 — data discovery, leakage audit, preprocessing, and fixed partitions
from sklearn.model_selection import train_test_split

IMMUTABLE_PATTERNS = [
    r"(^|_)ip($|_)", r"(^|_)mac($|_)", r"timestamp", r"(^|_)time($|_)",
    r"(^|_)port($|_)", r"flow[_-]?id", r"(^|_)id($|_)", r"proto", r"service",
    r"conn[_-]?state", r"label", r"attack", r"category", r"target", r"class"
]

def is_immutable_name(name):
    # Normalize punctuation so fields such as tcp.srcport, ip.src_host,
    # and conn_state are treated consistently.
    n = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if any(re.search(pattern, n) for pattern in IMMUTABLE_PATTERNS):
        return True
    parts = set(n.split("_"))
    if parts.intersection({"ip", "mac", "timestamp", "time", "port", "proto", "service",
                           "label", "attack", "category", "target", "class", "id"}):
        return True
    # Covers tcp.srcport/dstport and flow identifiers after normalization.
    return n.endswith("srcport") or n.endswith("dstport") or "flow_id" in n


def deterministic_cap(df, y, family, max_rows, seed=2026):
    if max_rows is None or len(df) <= max_rows:
        return df.reset_index(drop=True), np.asarray(y), np.asarray(family)
    rng = np.random.default_rng(seed)
    groups = pd.DataFrame({"y": y, "family": family}).astype(str)
    groups["group"] = groups["y"] + "::" + groups["family"]
    keep = []
    frac = max_rows / len(df)
    for _, idx in groups.groupby("group").groups.items():
        idx = np.asarray(list(idx))
        n = max(1, min(len(idx), int(round(len(idx) * frac))))
        keep.extend(rng.choice(idx, size=n, replace=False).tolist())
    if len(keep) > max_rows:
        keep = rng.choice(np.asarray(keep), size=max_rows, replace=False).tolist()
    keep = np.sort(np.asarray(keep))
    return df.iloc[keep].reset_index(drop=True), np.asarray(y)[keep], np.asarray(family)[keep]


def prepare_dataset(name, path):
    print(f"Loading {name} in memory-safe chunks: {path}")
    CHUNK_ROWS = 50_000
    probe = pd.read_csv(path, nrows=1000, low_memory=False)
    original_shape = [None, len(probe.columns)]
    label_col = choose_label_column(probe.columns)
    family_col = choose_family_column(probe.columns, label_col)

    # Infer the numerical traffic feature map from a small probe only.
    candidate_cols = []
    dropped = {}
    for c in probe.columns:
        if c in {label_col, family_col} or is_immutable_name(c):
            dropped[c] = "label_or_immutable/provenance_name"
            continue
        numeric = pd.to_numeric(probe[c], errors="coerce")
        if numeric.notna().mean() < 0.90:
            dropped[c] = "non_numeric_or_sparse"
            continue
        candidate_cols.append(c)
    if not candidate_cols:
        raise ValueError(f"No numerical traffic features retained for {name}")

    # First pass: count label/family groups without loading feature columns.
    count_cols = list(dict.fromkeys([label_col, family_col]))
    group_counts = {}
    total_rows = 0
    for chunk in pd.read_csv(path, usecols=count_cols, chunksize=CHUNK_ROWS, low_memory=False):
        y_chunk = label_to_binary(chunk[label_col])
        family_chunk = chunk[family_col].astype(str).fillna("unknown").to_numpy()
        keys = np.asarray([f"{int(y)}::{f}" for y, f in zip(y_chunk, family_chunk)], dtype=object)
        values, counts = np.unique(keys, return_counts=True)
        for key, count in zip(values, counts):
            group_counts[str(key)] = group_counts.get(str(key), 0) + int(count)
        total_rows += len(chunk)

    original_shape[0] = int(total_rows)
    if MAX_ROWS_PER_DATASET is None or total_rows <= MAX_ROWS_PER_DATASET:
        quotas = {k: int(v) for k, v in group_counts.items()}
    else:
        cap = int(MAX_ROWS_PER_DATASET)
        quotas = {k: max(1, int(round(cap * v / total_rows))) for k, v in group_counts.items()}
        # Adjust rounding while retaining at least one record per group.
        while sum(quotas.values()) > cap:
            key = max((k for k in quotas if quotas[k] > 1), key=lambda k: quotas[k], default=None)
            if key is None:
                break
            quotas[key] -= 1
        while sum(quotas.values()) < cap:
            key = max((k for k in quotas if quotas[k] < group_counts[k]),
                      key=lambda k: group_counts[k] - quotas[k], default=None)
            if key is None:
                break
            quotas[key] += 1

    # Second pass: read only the selected columns and sample each group deterministically.
    usecols = list(dict.fromkeys([label_col, family_col] + candidate_cols))
    parts = []
    taken = {k: 0 for k in quotas}
    rng = np.random.default_rng(SPLIT_SEED)
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=CHUNK_ROWS, low_memory=False):
        y_chunk = label_to_binary(chunk[label_col])
        family_chunk = chunk[family_col].astype(str).fillna("unknown").to_numpy()
        keys = np.asarray([f"{int(y)}::{f}" for y, f in zip(y_chunk, family_chunk)], dtype=object)
        X_chunk = chunk[candidate_cols].apply(pd.to_numeric, errors="coerce")
        X_chunk = X_chunk.replace([np.inf, -np.inf], np.nan)
        for key in np.unique(keys):
            key = str(key)
            if key not in quotas or taken[key] >= quotas[key]:
                continue
            idx = np.flatnonzero(keys == key)
            remaining = quotas[key] - taken[key]
            # Proportional chunk sampling avoids retaining the first attack block
            # when the file is ordered by attack family.
            n_take = min(len(idx), remaining, max(1, int(round(len(idx) * quotas[key] / group_counts[key]))))
            chosen = idx if n_take == len(idx) else rng.choice(idx, size=n_take, replace=False)
            sub = X_chunk.iloc[np.asarray(chosen)].copy()
            sub["__y__"] = y_chunk[np.asarray(chosen)]
            sub["__family__"] = family_chunk[np.asarray(chosen)]
            parts.append(sub)
            taken[key] += int(n_take)

    if not parts:
        raise RuntimeError(f"No rows were retained from {path}")
    sampled = pd.concat(parts, ignore_index=True)
    sampled["__group__"] = sampled["__y__"].astype(str) + "::" + sampled["__family__"].astype(str)
    # Enforce the quotas exactly after chunk rounding.
    final_parts = []
    for key, group in sampled.groupby("__group__", sort=False):
        n_take = min(len(group), quotas.get(str(key), len(group)))
        final_parts.append(group.sample(n=n_take, random_state=SPLIT_SEED) if len(group) > n_take else group)
    sampled = pd.concat(final_parts, ignore_index=True)
    y = sampled.pop("__y__").to_numpy(dtype=np.int64)
    family = sampled.pop("__family__").astype(str).to_numpy()
    sampled.pop("__group__")
    Xdf = sampled[candidate_cols].replace([np.inf, -np.inf], np.nan)
    all_nan = [c for c in Xdf.columns if Xdf[c].isna().all()]
    for c in all_nan:
        dropped[c] = "all_nan"
    Xdf = Xdf.drop(columns=all_nan)
    X = Xdf.to_numpy(dtype=np.float32)

    # Fixed dataset-level split; no seed-dependent resampling.
    indices = np.arange(len(y))
    train_idx, temp_idx = train_test_split(indices, test_size=0.30, stratify=y, random_state=SPLIT_SEED)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=y[temp_idx], random_state=SPLIT_SEED)

    # Fit robust quantile scaling on train only.
    q01 = np.nanquantile(X[train_idx], 0.01, axis=0)
    q99 = np.nanquantile(X[train_idx], 0.99, axis=0)
    med = np.nanmedian(X[train_idx], axis=0)
    denom = q99 - q01
    keep = denom > 1e-12
    feature_cols = [c for c, k in zip(Xdf.columns, keep) if k]
    X = X[:, keep]
    q01, q99, med = q01[keep], q99[keep], med[keep]
    X = np.where(np.isnan(X), med[None, :], X)
    X = np.clip((X - q01[None, :]) / denom[keep][None, :], 0.0, 1.0).astype(np.float32)

    # Fixed explanation subset: equal benign/malicious where possible.
    y = y.astype(np.int64)
    y_test = y[test_idx]
    benign = test_idx[y_test == 0]
    malicious = test_idx[y_test == 1]
    rng = np.random.default_rng(SPLIT_SEED)
    n_b = min(EXPLAIN_BENIGN_ROWS, len(benign))
    n_m = min(EXPLAIN_MALICIOUS_ROWS, len(malicious))
    eval_ids = np.concatenate([
        np.sort(rng.choice(benign, n_b, replace=False)),
        np.sort(rng.choice(malicious, n_m, replace=False))
    ])
    attack_ids = eval_ids[y[eval_ids] == 1][:min(ATTACK_ROWS, n_m)]

    out = DIRS["processed"] / name
    out.mkdir(parents=True, exist_ok=True)
    # Convert global row indices to positions in the frozen local test array.
    test_pos = {int(global_id): int(local_id) for local_id, global_id in enumerate(test_idx)}
    np.savez_compressed(
        out / "data.npz",
        X_train=X[train_idx], y_train=y[train_idx],
        X_val=X[val_idx], y_val=y[val_idx],
        X_test=X[test_idx], y_test=y[test_idx],
        eval_ids=np.asarray([test_pos[int(i)] for i in eval_ids], dtype=np.int64),
        attack_ids=np.asarray([test_pos[int(i)] for i in attack_ids], dtype=np.int64),
    )
    meta = {
        "name": name, "source_path": str(path), "original_shape": original_shape,
        "rows_after_cap": int(len(y)), "label_column": label_col, "family_column": family_col,
        "feature_columns": feature_cols, "dropped_columns": dropped,
        "n_features": len(feature_cols),
        "train_rows": int(len(train_idx)), "val_rows": int(len(val_idx)), "test_rows": int(len(test_idx)),
        "train_positive_rate": float(y[train_idx].mean()), "test_positive_rate": float(y[test_idx].mean()),
        "q01": q01.tolist(), "q99": q99.tolist(), "median": med.tolist(),
        "eval_rows": int(len(eval_ids)), "attack_rows": int(len(attack_ids)),
        "split_seed": SPLIT_SEED, "feature_policy": "numeric traffic features; provenance/identifier names excluded",
        "chunk_rows": CHUNK_ROWS, "rows_before_cap": int(total_rows),
    }
    save_json(out / "metadata.json", meta)
    pd.DataFrame({"dataset": name, "split": ["train", "validation", "test"], "rows": [len(train_idx), len(val_idx), len(test_idx)]}).to_csv(out / "split_summary.csv", index=False)
    return out, meta


selected = {"ton": choose_input("ton"), "edge": choose_input("edge")}
print("Selected inputs:", selected)
prepared = {}
metadata = {}
for name, path in selected.items():
    prepared[name], metadata[name] = prepare_dataset(name, path)

record_resource("stage_01_data_audit", DIRS["processed"])
manifest = write_manifest(DIRS["processed"], "stage_01_data_audit", {"selected_inputs": {k: str(v) for k, v in selected.items()}})
zip_stage("stage_01_data_audit", [DIRS["processed"]])


## Stage 2 — reproduce and train the MLP baseline

The clean condition is the primary reproduced baseline. The adversarial-training condition is a secondary control. Checkpoints and per-epoch histories are saved so the stage can be inspected without rerunning training.

In [ ]:
# Stage 2 — MLP training, clean baseline and adversarial-training control
class IDSMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=HIDDEN_DIMS, dropout=DROPOUT):
        super().__init__()
        layers = []
        d = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)


def predict_prob(model, X, batch_size=8192):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).to(DEVICE)
            out.append(torch.sigmoid(model(xb)).detach().cpu().numpy())
    return np.concatenate(out)


def classification_metrics(y, p, threshold=0.5):
    pred = (p >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "positive_f1": f1_score(y, pred, zero_division=0),
        "pr_auc": average_precision_score(y, p),
        "roc_auc": roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
        "false_positive_rate": confusion_matrix(y, pred, labels=[0,1])[0,1] / max(1, (y == 0).sum()),
        "false_negative_rate": confusion_matrix(y, pred, labels=[0,1])[1,0] / max(1, (y == 1).sum()),
    }


def fgsm_training_batch(model, xb, yb, loss_fn, eps=0.01):
    x_adv = xb.detach().clone().requires_grad_(True)
    loss = loss_fn(model(x_adv), yb)
    grad = torch.autograd.grad(loss, x_adv, only_inputs=True)[0]
    return (x_adv + eps * grad.sign()).clamp(0, 1).detach()


def train_one(Xtr, ytr, Xv, yv, seed, condition, dataset):
    set_seed(seed)
    model = IDSMLP(Xtr.shape[1]).to(DEVICE)
    pos_weight = torch.tensor([(len(ytr) - ytr.sum()) / max(1, ytr.sum())], dtype=torch.float32, device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr.astype(np.float32))),
                        batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    best_state, best_score, wait = None, -np.inf, 0
    history = []
    start = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            if condition == "advtrain":
                x_use = fgsm_training_batch(model, xb, yb, loss_fn, eps=min(EPSILONS))
            else:
                x_use = xb
            loss = loss_fn(model(x_use), yb)
            loss.backward()
            opt.step()
            train_loss += float(loss.detach().cpu()) * len(yb)
        p_val = predict_prob(model, Xv)
        val_m = classification_metrics(yv, p_val)
        row = {"dataset": dataset, "condition": condition, "seed": seed, "epoch": epoch,
               "train_loss": train_loss / len(Xtr), **{f"val_{k}": v for k, v in val_m.items()}}
        history.append(row)
        if val_m["macro_f1"] > best_score:
            best_score = val_m["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state)
    elapsed = time.time() - start
    return model, pd.DataFrame(history), elapsed

model_rows, history_rows = [], []
conditions = ["clean"] + (["advtrain"] if RUN_ADVERSARIAL_TRAINING else [])
for dataset in ["ton", "edge"]:
    z = np.load(prepared[dataset] / "data.npz")
    for condition in conditions:
        for seed in SEEDS:
            print(f"Training dataset={dataset} condition={condition} seed={seed}")
            model, hist, elapsed = train_one(z["X_train"], z["y_train"], z["X_val"], z["y_val"], seed, condition, dataset)
            model_path = DIRS["models"] / f"mlp_{dataset}_{condition}_seed{seed}.pt"
            torch.save(model.state_dict(), model_path)
            hist.to_csv(DIRS["models"] / f"history_{dataset}_{condition}_seed{seed}.csv", index=False)
            p_test = predict_prob(model, z["X_test"])
            met = classification_metrics(z["y_test"], p_test)
            model_rows.append({"dataset": dataset, "condition": condition, "seed": seed, "train_seconds": elapsed, **met})
            history_rows.extend(hist.to_dict("records"))

pd.DataFrame(model_rows).to_csv(DIRS["models"] / "baseline_metrics.csv", index=False)
pd.DataFrame(history_rows).to_csv(DIRS["models"] / "training_histories.csv", index=False)
record_resource("stage_02_baseline_training", DIRS["models"])
write_manifest(DIRS["models"], "stage_02_baseline_training")
zip_stage("stage_02_baseline_training", [DIRS["models"]])


## Stage 3 — constrained FGSM/PGD evasion

The attack modifies only the normalized numerical traffic features retained by Stage 1. Malicious-to-benign attack success is evaluated only for samples that the clean model initially classified as malicious.

In [ ]:
# Stage 3 — constrained FGSM and PGD attacks

def attack_fgsm(model, x, eps):
    model.eval()
    xa = x.detach().clone().requires_grad_(True)
    p_mal = torch.sigmoid(model(xa)).mean()
    grad = torch.autograd.grad(p_mal, xa)[0]
    # Minimize malicious probability: targeted malicious -> benign evasion.
    return (xa - eps * grad.sign()).clamp(0, 1).detach()


def attack_pgd(model, x, eps, steps=PGD_STEPS, random_start=True, attack_seed=0):
    model.eval()
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(int(attack_seed))
    if random_start:
        delta = (torch.rand(x.shape, generator=gen, device=DEVICE) * 2.0 - 1.0) * eps
        xa = (x + delta).clamp(0, 1).detach()
    else:
        xa = x.detach().clone()
    alpha = eps / max(1, steps)
    for _ in range(steps):
        xa.requires_grad_(True)
        p_mal = torch.sigmoid(model(xa)).mean()
        grad = torch.autograd.grad(p_mal, xa)[0]
        xa = xa - alpha * grad.sign()
        xa = torch.max(torch.min(xa, x + eps), x - eps).clamp(0, 1).detach()
    return xa

attack_rows = []
for dataset in ["ton", "edge"]:
    z = np.load(prepared[dataset] / "data.npz")
    meta = metadata[dataset]
    ids = z["attack_ids"].astype(int)
    X_attack = z["X_test"][ids]
    y_attack = z["y_test"][ids]
    for condition in conditions:
        for seed in SEEDS:
            model = IDSMLP(z["X_train"].shape[1]).to(DEVICE)
            model.load_state_dict(torch.load(DIRS["models"] / f"mlp_{dataset}_{condition}_seed{seed}.pt", map_location=DEVICE))
            x = torch.from_numpy(X_attack).float().to(DEVICE)
            p_clean = predict_prob(model, X_attack)
            clean_pred = (p_clean >= 0.5).astype(int)
            eligible = (y_attack == 1) & (clean_pred == 1)
            arrays = {"X_clean": X_attack.astype(np.float32), "p_clean": p_clean.astype(np.float32), "y": y_attack.astype(np.int64)}
            for attack_name in ["fgsm", "pgd"]:
                for eps in EPSILONS:
                    start = time.time()
                    if attack_name == "fgsm":
                        xa = attack_fgsm(model, x, eps)
                    else:
                        xa = attack_pgd(model, x, eps, attack_seed=seed + int(eps * 10000))
                    X_adv = xa.detach().cpu().numpy().astype(np.float32)
                    p_adv = predict_prob(model, X_adv)
                    pred_adv = (p_adv >= 0.5).astype(int)
                    success = eligible & (pred_adv == 0)
                    key = f"{attack_name}_eps{eps:.2f}"
                    arrays[f"X_{key}"] = X_adv
                    arrays[f"p_{key}"] = p_adv.astype(np.float32)
                    attack_rows.append({
                        "dataset": dataset, "condition": condition, "seed": seed,
                        "attack": attack_name, "epsilon": eps,
                        "eligible_count": int(eligible.sum()),
                        "success_count": int(success.sum()),
                        "attack_success_rate": float(success.sum() / max(1, eligible.sum())),
                        "malicious_recall_clean": float((clean_pred[y_attack == 1] == 1).mean()),
                        "malicious_recall_adv": float((pred_adv[y_attack == 1] == 1).mean()),
                        "mean_linf": float(np.abs(X_adv - X_attack).max(axis=1).mean()),
                        "attack_seconds": time.time() - start,
                    })
            np.savez_compressed(DIRS["attacks"] / f"attacks_{dataset}_{condition}_seed{seed}.npz", **arrays)

pd.DataFrame(attack_rows).to_csv(DIRS["attacks"] / "attack_metrics.csv", index=False)
record_resource("stage_03_attacks", DIRS["attacks"])
write_manifest(DIRS["attacks"], "stage_03_attacks")
zip_stage("stage_03_attacks", [DIRS["attacks"]])


## Stage 4 — SHAP/LIME explanations and faithfulness/stability metrics

The explanation subset is fixed in Stage 1. SHAP and LIME are generated for clean records and for the attacked versions of the same malicious records. The implementation stores signed attribution arrays, runtime, comprehensiveness, sufficiency, and clean-to-adversarial top-10 Jaccard stability.

In [ ]:
# Stage 4 — SHAP/LIME explanations, faithfulness, and stability

def load_model(dataset, condition, seed, input_dim):
    model = IDSMLP(input_dim).to(DEVICE)
    model.load_state_dict(torch.load(DIRS["models"] / f"mlp_{dataset}_{condition}_seed{seed}.pt", map_location=DEVICE))
    model.eval()
    return model


def predict_fn_numpy(model, arr):
    arr = np.asarray(arr, dtype=np.float32)
    return np.column_stack([1 - predict_prob(model, arr), predict_prob(model, arr)])


def shap_attributions(model, X, background):
    class ProbabilityWrapper(nn.Module):
        def __init__(self, base):
            super().__init__(); self.base = base
        def forward(self, x):
            return torch.sigmoid(self.base(x)).reshape(-1, 1)
    wrapped = ProbabilityWrapper(model).to(DEVICE).eval()
    xb = torch.from_numpy(background.astype(np.float32)).to(DEVICE)
    xx = torch.from_numpy(X.astype(np.float32)).to(DEVICE)
    try:
        explainer = shap.DeepExplainer(wrapped, xb)
        vals = explainer.shap_values(xx)
    except Exception as exc:
        print("DeepExplainer fallback:", exc)
        explainer = shap.GradientExplainer(wrapped, xb)
        vals = explainer.shap_values(xx)
    if isinstance(vals, list):
        vals = vals[0]
    vals = np.asarray(vals)
    if vals.ndim == 3:
        vals = vals[:, :, 0]
    return vals.astype(np.float32)


def lime_attributions(model, X, background, feature_names, seed):
    explainer = LimeTabularExplainer(
        background.astype(np.float64), feature_names=list(feature_names),
        class_names=["benign", "malicious"], mode="classification",
        discretize_continuous=False, random_state=int(seed)
    )
    vals = np.zeros_like(X, dtype=np.float32)
    for i, row in enumerate(X):
        exp = explainer.explain_instance(row.astype(np.float64), lambda a: predict_fn_numpy(model, a),
                                         num_features=X.shape[1], num_samples=LIME_NUM_SAMPLES, labels=(1,))
        for j, v in exp.as_map().get(1, []):
            vals[i, int(j)] = float(v)
    return vals


def topk_indices(attr, k):
    return np.argsort(-np.abs(attr), axis=1)[:, :k]


def faithfulness_metrics(model, X, attr, reference):
    p = predict_prob(model, X)
    fractions = [0.05, 0.10, 0.20, 0.40]
    comps, suffs = [], []
    for frac in fractions:
        k = max(1, int(round(frac * X.shape[1])))
        top = topk_indices(attr, k)
        xd = X.copy(); xs = np.repeat(reference[None, :], len(X), axis=0)
        rows = np.arange(len(X))[:, None]
        xd[rows, top] = reference[top]
        xs[rows, top] = X[rows, top]
        comps.append(p - predict_prob(model, xd))
        suffs.append(predict_prob(model, xs))
    comp = np.stack(comps, axis=1)
    suff = np.stack(suffs, axis=1)
    return {
        "comprehensiveness_auc": float(np.trapezoid(comp.mean(axis=0), fractions)),
        "sufficiency_auc": float(np.trapezoid(suff.mean(axis=0), fractions)),
        "comprehensiveness_at_10pct": float(comp[:, 1].mean()),
        "sufficiency_at_10pct": float(suff[:, 1].mean()),
    }


def mean_jaccard(a, b, k=10):
    k = min(k, a.shape[1])
    ta, tb = topk_indices(a, k), topk_indices(b, k)
    return float(np.mean([
        len(set(ta[i]) & set(tb[i])) / max(1, len(set(ta[i]) | set(tb[i])))
        for i in range(len(ta))
    ]))

explanation_rows = []
for dataset in ["ton", "edge"]:
    z = np.load(prepared[dataset] / "data.npz")
    meta = metadata[dataset]
    eval_ids = z["eval_ids"].astype(int)
    attack_ids = z["attack_ids"].astype(int)
    X_eval = z["X_test"][eval_ids]
    X_attack = z["X_test"][attack_ids]
    eval_position = {int(row_id): int(pos) for pos, row_id in enumerate(eval_ids)}
    attack_eval_positions = np.asarray([eval_position[int(row_id)] for row_id in attack_ids], dtype=int)
    background = z["X_train"][np.linspace(0, len(z["X_train"]) - 1, min(SHAP_BACKGROUND_ROWS, len(z["X_train"])), dtype=int)]
    reference = np.median(background, axis=0).astype(np.float32)
    for condition in conditions:
        for seed in SEEDS:
            model = load_model(dataset, condition, seed, z["X_train"].shape[1])
            # Clean explanations on a fixed benign/malicious evaluation subset.
            for explainer_name in ["SHAP", "LIME"]:
                t0 = time.time()
                if explainer_name == "SHAP":
                    attr_clean = shap_attributions(model, X_eval, background)
                else:
                    attr_clean = lime_attributions(model, X_eval, background, meta["feature_columns"], seed + 1000)
                fm = faithfulness_metrics(model, X_eval, attr_clean, reference)
                explanation_rows.append({
                    "dataset": dataset, "condition": condition, "seed": seed,
                    "explainer": explainer_name, "view": "clean", "attack": "clean", "epsilon": 0.0,
                    "runtime_seconds": time.time() - t0, **fm,
                })
                # Generate clean and attacked explanations for identical malicious records.
                attack_file = np.load(DIRS["attacks"] / f"attacks_{dataset}_{condition}_seed{seed}.npz")
                attr_attack_clean = attr_clean[attack_eval_positions]
                # The fixed attack IDs are the malicious tail of the fixed eval subset by construction.
                for eps in EPSILONS:
                    key = f"pgd_eps{eps:.2f}"
                    X_adv = attack_file[f"X_{key}"]
                    t1 = time.time()
                    if explainer_name == "SHAP":
                        attr_adv = shap_attributions(model, X_adv, background)
                    else:
                        attr_adv = lime_attributions(model, X_adv, background, meta["feature_columns"], seed + 2000 + int(eps * 1000))
                    fm_adv = faithfulness_metrics(model, X_adv, attr_adv, reference)
                    explanation_rows.append({
                        "dataset": dataset, "condition": condition, "seed": seed,
                        "explainer": explainer_name, "view": "adversarial", "attack": "pgd", "epsilon": eps,
                        "runtime_seconds": time.time() - t1,
                        "clean_adv_jaccard_top10": mean_jaccard(attr_attack_clean, attr_adv, k=10),
                        **fm_adv,
                    })
                    np.savez_compressed(
                        DIRS["explanations"] / f"attr_{dataset}_{condition}_{explainer_name.lower()}_seed{seed}_eps{eps:.2f}.npz",
                        attr_clean=attr_attack_clean, attr_adv=attr_adv,
                    )

pd.DataFrame(explanation_rows).to_csv(DIRS["explanations"] / "explanation_metrics.csv", index=False)
record_resource("stage_04_explanations", DIRS["explanations"])
write_manifest(DIRS["explanations"], "stage_04_explanations")
zip_stage("stage_04_explanations", [DIRS["explanations"]])


## Stage 5 — seed-level inference and Springer-compatible PDF figures

The primary inferential unit is one aggregated metric per training seed. Figures use vector PDF output, embedded TrueType-compatible fonts, serif typography, compact Springer-style dimensions, restrained color, and accessible labels. One-column figures use approximately 84 mm width; two-column figures use approximately 174 mm width.

In [ ]:
# Stage 5 — statistical summaries and vector PDF figures
from itertools import product

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.6,
    "lines.linewidth": 1.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.transparent": False,
})
COLORS = {"SHAP": "#0072B2", "LIME": "#D55E00", "clean": "#009E73", "advtrain": "#CC79A7"}

def save_pdf(fig, name, width_mm=84, height_mm=62):
    path = DIRS["figures"] / f"{name}.pdf"
    fig.set_size_inches(width_mm / 25.4, height_mm / 25.4)
    fig.savefig(path, format="pdf", bbox_inches="tight", pad_inches=0.03, metadata={"Creator": "Kaggle IoT/IIoT IDS experiment"})
    plt.close(fig)
    return path

attack_df = pd.read_csv(DIRS["attacks"] / "attack_metrics.csv")
expl_df = pd.read_csv(DIRS["explanations"] / "explanation_metrics.csv")
model_df = pd.read_csv(DIRS["models"] / "baseline_metrics.csv")

# Exact paired sign-permutation p-value over the five seed differences.
def exact_sign_pvalue(values):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan
    observed = abs(vals.mean())
    stats = []
    for signs in product([-1, 1], repeat=len(vals)):
        stats.append(abs((vals * np.asarray(signs)).mean()))
    return float(np.mean(np.asarray(stats) >= observed - 1e-12))


def bootstrap_seed_ci(values, n_boot=5000, seed=2026):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = np.array([rng.choice(vals, len(vals), replace=True).mean() for _ in range(n_boot)])
    return float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))


def paired_effect_statistics(values):
    """Cohen's dz and small-sample Hedges correction for paired seed differences."""
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    n = len(vals)
    if n == 0:
        return {"cohens_dz": np.nan, "hedges_gz": np.nan}
    sd = float(np.std(vals, ddof=1)) if n > 1 else np.nan
    dz = float(np.mean(vals) / sd) if np.isfinite(sd) and sd > 0 else np.nan
    correction = (1.0 - 3.0 / (4.0 * n - 5.0)) if n > 2 else np.nan
    return {"cohens_dz": dz, "hedges_gz": float(correction * dz) if np.isfinite(dz) and np.isfinite(correction) else np.nan}


def bootstrap_mean_ci(values, n_boot=5000, seed=2026):
    """Bootstrap a mean while treating each supplied value as one independent cluster."""
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    sample_idx = rng.integers(0, len(vals), size=(n_boot, len(vals)))
    boot = vals[sample_idx].mean(axis=1)
    return float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))


def bootstrap_independent_difference(clean_values, adv_values, n_boot=5000, seed=2026):
    """Naive independent-row bootstrap used as a sensitivity comparator."""
    x = np.asarray(clean_values, dtype=float); y = np.asarray(adv_values, dtype=float)
    x = x[np.isfinite(x)]; y = y[np.isfinite(y)]
    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    bx = x[rng.integers(0, len(x), size=(n_boot, len(x)))].mean(axis=1)
    by = y[rng.integers(0, len(y), size=(n_boot, len(y)))].mean(axis=1)
    boot = by - bx
    return float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))

# Aggregate seed-level summaries.
seed_summary = expl_df.groupby(["dataset", "condition", "explainer", "view", "attack", "epsilon"], dropna=False).agg(
    comprehensiveness_auc_mean=("comprehensiveness_auc", "mean"),
    comprehensiveness_auc_sd=("comprehensiveness_auc", "std"),
    sufficiency_auc_mean=("sufficiency_auc", "mean"),
    jaccard_mean=("clean_adv_jaccard_top10", "mean"),
    runtime_mean=("runtime_seconds", "mean"),
    n_seeds=("seed", "nunique"),
).reset_index()
seed_summary.to_csv(DIRS["tables"] / "seed_level_explanation_summary.csv", index=False)
attack_df.to_csv(DIRS["tables"] / "attack_metrics_by_seed.csv", index=False)
model_df.to_csv(DIRS["tables"] / "model_metrics_by_seed.csv", index=False)

# SHAP-vs-LIME paired differences for common groups.
stat_rows = []
for keys, group in expl_df.groupby(["dataset", "condition", "view", "attack", "epsilon"], dropna=False):
    pivot = group.pivot_table(index="seed", columns="explainer", values="comprehensiveness_auc", aggfunc="mean")
    if {"SHAP", "LIME"}.issubset(pivot.columns):
        diff = (pivot["SHAP"] - pivot["LIME"]).dropna().to_numpy()
        lo, hi = bootstrap_seed_ci(diff)
        effect = paired_effect_statistics(diff)
        try:
            w_p = float(wilcoxon(diff).pvalue) if len(diff) >= 3 and np.any(diff != 0) else np.nan
        except Exception:
            w_p = np.nan
        stat_rows.append({"dataset": keys[0], "condition": keys[1], "view": keys[2], "attack": keys[3], "epsilon": keys[4],
                          "comparison": "SHAP_minus_LIME_comprehensiveness_AUC", "mean_difference": float(np.mean(diff)),
                          "median_difference": float(np.median(diff)), "ci95_low": lo, "ci95_high": hi,
                          "exact_sign_p": exact_sign_pvalue(diff), "wilcoxon_p": w_p,
                          **effect, "n_seeds": len(diff)})
stats_df = pd.DataFrame(stat_rows)
# Holm correction within each dataset/condition/view/attack family.
def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    finite = np.isfinite(pvalues)
    out = np.full(len(pvalues), np.nan)
    if finite.any():
        order = np.argsort(pvalues[finite])
        sorted_p = pvalues[finite][order]
        adjusted = np.maximum.accumulate((len(sorted_p) - np.arange(len(sorted_p))) * sorted_p)
        adjusted = np.minimum(adjusted, 1.0)
        tmp = np.empty(len(sorted_p)); tmp[order] = adjusted
        out[np.where(finite)[0]] = tmp
    return out
if len(stats_df):
    stats_df["holm_wilcoxon_p"] = np.nan
    for _, idx in stats_df.groupby(["dataset", "condition", "view", "attack"], dropna=False).groups.items():
        stats_df.loc[idx, "holm_wilcoxon_p"] = holm_adjust(stats_df.loc[idx, "wilcoxon_p"].to_numpy())
stats_df.to_csv(DIRS["stats"] / "paired_seed_statistics.csv", index=False)

# Condition contrasts: adversarial training minus clean, paired by training seed.
# These rows provide standardized paired effects for baseline and attack outcomes.
condition_effect_rows = []
for (dataset, attack, epsilon), group in attack_df.groupby(["dataset", "attack", "epsilon"]):
    piv = group.pivot_table(index="seed", columns="condition", values=["attack_success_rate", "malicious_recall_adv"], aggfunc="mean")
    if not {"clean", "advtrain"}.issubset(set(piv.columns.get_level_values(1))):
        continue
    for metric in ["attack_success_rate", "malicious_recall_adv"]:
        if (metric, "clean") not in piv or (metric, "advtrain") not in piv:
            continue
        diff = (piv[(metric, "advtrain")] - piv[(metric, "clean")]).dropna().to_numpy()
        eff = paired_effect_statistics(diff)
        lo, hi = bootstrap_seed_ci(diff, seed=2026 + len(condition_effect_rows))
        try:
            w_p = float(wilcoxon(diff).pvalue) if len(diff) >= 3 and np.any(diff != 0) else np.nan
        except Exception:
            w_p = np.nan
        condition_effect_rows.append({
            "comparison_family": "attack_condition_contrast",
            "dataset": dataset, "attack": attack, "epsilon": epsilon,
            "metric": metric, "comparison": "advtrain_minus_clean",
            "mean_difference": float(np.mean(diff)), "median_difference": float(np.median(diff)),
            "ci95_low": lo, "ci95_high": hi, "exact_sign_p": exact_sign_pvalue(diff),
            "wilcoxon_p": w_p, **eff, "n_clusters": len(diff), "cluster_unit": "training_seed",
        })

for dataset, group in model_df.groupby("dataset"):
    piv = group.pivot_table(index="seed", columns="condition", values=["macro_f1", "balanced_accuracy", "pr_auc"], aggfunc="mean")
    for metric in ["macro_f1", "balanced_accuracy", "pr_auc"]:
        if (metric, "clean") not in piv or (metric, "advtrain") not in piv:
            continue
        diff = (piv[(metric, "advtrain")] - piv[(metric, "clean")]).dropna().to_numpy()
        eff = paired_effect_statistics(diff)
        lo, hi = bootstrap_seed_ci(diff, seed=3026 + len(condition_effect_rows))
        try:
            w_p = float(wilcoxon(diff).pvalue) if len(diff) >= 3 and np.any(diff != 0) else np.nan
        except Exception:
            w_p = np.nan
        condition_effect_rows.append({
            "comparison_family": "baseline_condition_contrast",
            "dataset": dataset, "attack": "clean", "epsilon": 0.0,
            "metric": metric, "comparison": "advtrain_minus_clean",
            "mean_difference": float(np.mean(diff)), "median_difference": float(np.median(diff)),
            "ci95_low": lo, "ci95_high": hi, "exact_sign_p": exact_sign_pvalue(diff),
            "wilcoxon_p": w_p, **eff, "n_clusters": len(diff), "cluster_unit": "training_seed",
        })

condition_effect_df = pd.DataFrame(condition_effect_rows)
if len(condition_effect_df):
    condition_effect_df["holm_wilcoxon_p"] = np.nan
    for _, idx in condition_effect_df.groupby(["comparison_family", "dataset", "metric", "attack"], dropna=False).groups.items():
        condition_effect_df.loc[idx, "holm_wilcoxon_p"] = holm_adjust(condition_effect_df.loc[idx, "wilcoxon_p"].to_numpy())
condition_effect_df.to_csv(DIRS["stats"] / "condition_effect_sizes.csv", index=False)

# Clustered-bootstrap sensitivity: resample complete training-seed pairs, and compare
# with a deliberately naive independent-condition bootstrap. This preserves the
# pairing induced by each trained seed while exposing sensitivity to that choice.
cluster_rows = []
for (dataset, attack, epsilon), group in attack_df.groupby(["dataset", "attack", "epsilon"]):
    piv = group.pivot_table(index="seed", columns="condition", values=["attack_success_rate", "malicious_recall_adv"], aggfunc="mean")
    for metric in ["attack_success_rate", "malicious_recall_adv"]:
        if (metric, "clean") not in piv or (metric, "advtrain") not in piv:
            continue
        clean_vals = piv[(metric, "clean")].dropna().to_numpy()
        adv_vals = piv[(metric, "advtrain")].dropna().to_numpy()
        paired_diff = adv_vals - clean_vals
        cluster_lo, cluster_hi = bootstrap_mean_ci(paired_diff, seed=4026 + len(cluster_rows))
        naive_lo, naive_hi = bootstrap_independent_difference(clean_vals, adv_vals, seed=5026 + len(cluster_rows))
        cluster_rows.append({
            "dataset": dataset, "attack": attack, "epsilon": epsilon, "metric": metric,
            "contrast": "advtrain_minus_clean", "cluster_unit": "training_seed",
            "n_clusters": len(paired_diff), "point_estimate": float(np.mean(paired_diff)),
            "clustered_bootstrap_ci95_low": cluster_lo, "clustered_bootstrap_ci95_high": cluster_hi,
            "independent_bootstrap_ci95_low": naive_lo, "independent_bootstrap_ci95_high": naive_hi,
            "clustered_interval_width": float(cluster_hi - cluster_lo),
            "independent_interval_width": float(naive_hi - naive_lo), "n_boot": 5000,
        })
cluster_df = pd.DataFrame(cluster_rows)
cluster_df.to_csv(DIRS["stats"] / "clustered_bootstrap_sensitivity.csv", index=False)

# Compact figure captions and provenance metadata for manuscript use.
figure_captions = {
    "fig_attack_success_pgd.pdf": "PGD attack success rate by normalized perturbation budget; points are means across five training seeds and error bars are seed standard deviations.",
    "fig_faithfulness_comprehensiveness.pdf": "Distribution of comprehensiveness AUC for SHAP and tabular LIME on clean and adversarial views.",
    "fig_explanation_stability_jaccard.pdf": "Top-10 explanation stability measured by clean-to-adversarial Jaccard overlap across PGD budgets.",
    "fig_explanation_runtime.pdf": "Mean explanation runtime by dataset and explainer; values aggregate clean and adversarial explanation calls.",
}
(DIRS["figures"] / "figure_captions.json").write_text(json.dumps(figure_captions, indent=2))
record_resource("stage_05_figures_statistics", DIRS["stats"])

# Figure 1: attack success by dataset, condition, attack, and budget.
fig, ax = plt.subplots()
plot = attack_df[attack_df["attack"] == "pgd"].copy()
if len(plot):
    for (dataset, condition), g in plot.groupby(["dataset", "condition"]):
        agg = g.groupby("epsilon")["attack_success_rate"].agg(["mean", "std"]).reset_index()
        x = agg["epsilon"].to_numpy(); y = agg["mean"].to_numpy(); e = agg["std"].fillna(0).to_numpy()
        ax.errorbar(x, y, yerr=e, marker="o", capsize=2, label=f"{dataset}-{condition}")
ax.set_xlabel("Normalized PGD budget $\\epsilon$")
ax.set_ylabel("Attack success rate")
ax.set_ylim(0, 1.05); ax.yaxis.set_major_locator(MaxNLocator(5))
ax.legend(frameon=False, loc="best")
ax.spines[["top", "right"]].set_visible(False)
fig_attack = save_pdf(fig, "fig_attack_success_pgd", 84, 62)

# Figure 2: comprehensiveness AUC for clean and attacked views.
fig, ax = plt.subplots()
plot = expl_df.copy()
if len(plot):
    groups = []
    labels = []
    for (dataset, view, explainer), g in plot.groupby(["dataset", "view", "explainer"]):
        groups.append(g["comprehensiveness_auc"].dropna().to_numpy())
        labels.append(f"{dataset}\n{view}\n{explainer}")
    if groups:
        positions = np.arange(len(groups))
        ax.boxplot(groups, positions=positions, widths=0.55, patch_artist=True,
                   boxprops={"facecolor": "#D9D9D9", "linewidth": 0.6},
                   medianprops={"color": "black", "linewidth": 0.8})
        ax.set_xticks(positions); ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_ylabel("Comprehensiveness AUC")
ax.spines[["top", "right"]].set_visible(False)
fig_faith = save_pdf(fig, "fig_faithfulness_comprehensiveness", 174, 78)

# Figure 3: clean-to-adversarial top-10 Jaccard stability.
fig, ax = plt.subplots()
plot = expl_df[expl_df["view"] == "adversarial"].copy()
if len(plot):
    for (dataset, explainer), g in plot.groupby(["dataset", "explainer"]):
        agg = g.groupby("epsilon")["clean_adv_jaccard_top10"].agg(["mean", "std"]).reset_index()
        ax.errorbar(agg["epsilon"], agg["mean"], yerr=agg["std"].fillna(0), marker="o", capsize=2,
                    color=COLORS.get(explainer), label=f"{dataset}-{explainer}")
ax.set_xlabel("Normalized PGD budget $\\epsilon$")
ax.set_ylabel("Clean--adversarial top-10 Jaccard")
ax.set_ylim(0, 1.05); ax.yaxis.set_major_locator(MaxNLocator(5))
ax.legend(frameon=False, loc="best")
ax.spines[["top", "right"]].set_visible(False)
fig_stability = save_pdf(fig, "fig_explanation_stability_jaccard", 84, 62)

# Figure 4: explanation runtime.
fig, ax = plt.subplots()
if len(expl_df):
    rt = expl_df.groupby(["dataset", "explainer"])["runtime_seconds"].mean().reset_index()
    labels = [f"{r.dataset}\n{r.explainer}" for r in rt.itertuples()]
    ax.bar(np.arange(len(rt)), rt["runtime_seconds"], color=[COLORS.get(e, "#777777") for e in rt["explainer"]])
    ax.set_xticks(np.arange(len(rt))); ax.set_xticklabels(labels)
ax.set_ylabel("Mean explanation time (s)")
ax.spines[["top", "right"]].set_visible(False)
fig_runtime = save_pdf(fig, "fig_explanation_runtime", 84, 62)

write_manifest(DIRS["figures"], "stage_05_figures", {"figures": [str(p) for p in [fig_attack, fig_faith, fig_stability, fig_runtime]]})
write_manifest(DIRS["tables"], "stage_05_tables")
write_manifest(DIRS["stats"], "stage_05_statistics")
zip_stage("stage_05_statistics_figures", [DIRS["stats"], DIRS["tables"], DIRS["figures"]])


## Stage 6 — final experiment bundle

This ZIP is the single downloadable artifact for the Kaggle run. It contains each stage ZIP plus the final PDF figures, CSV tables, JSON metadata, attack summaries, explanation outputs, and model checkpoints.

In [ ]:
# Stage 6 — bundle all reproducible outputs and display the final Kaggle download link
final_zip = ROOT / "final_experiment_bundle.zip"
include = []
for p in sorted(ROOT.glob("stage_*.zip")):
    include.append(p)
for p in [DIRS["figures"], DIRS["tables"], DIRS["stats"], DIRS["processed"], DIRS["models"], DIRS["attacks"], DIRS["explanations"]]:
    include.append(p)
if (ROOT / "resource_metrics.csv").exists():
    include.append(ROOT / "resource_metrics.csv")
with zipfile.ZipFile(final_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in include:
        p = Path(p)
        if p.is_dir():
            for child in p.rglob("*"):
                if child.is_file():
                    zf.write(child, arcname=str(child.relative_to(ROOT)))
        elif p.exists():
            zf.write(p, arcname=p.name)

display(FileLink(str(final_zip), result_html_prefix="<b>Download final_experiment_bundle.zip:</b> "))
print("Final bundle:", final_zip, final_zip.stat().st_size, "bytes")


## Interpretation checklist

Before using any result in the paper, verify:

- the selected input paths and file hashes are recorded;
- the ToN-IoT clean baseline is compared with the published result;
- all five seeds completed for both datasets and both conditions;
- attack success is reported relative to correctly detected malicious records;
- immutable/provenance features were not modified by the attack;
- SHAP/LIME outputs exist for the same paired clean and attacked records;
- seed-level results, not only sample-level p-values, support the claims;
- every figure used in the manuscript is taken from `figures_pdf/*.pdf`;
- the final ZIP and each stage ZIP are downloaded from the Kaggle output panel.